
# SLAP2 session manifests and directory reorganization

This notebook provides a reviewable workflow for:

1. selecting **one session** or **all raw sessions for one subject ID**;
2. writing a tree manifest for each selected session;
3. building and validating a dry-run reorganization plan;
4. reviewing all files assigned to overflow;
5. moving files only after an explicit execution confirmation; and
6. writing post-reorganization manifests and operation reports.

The default layout is `nested`: the supplied raw session directory becomes a container holding the canonical raw, processed, and backup directories. Vascular reference images remain with raw data. Both `dendriticVoltageExtraction` and `ExperimentSummary` are routed to `source_extraction`.

> **Safety:** The notebook is dry-run by default. No files move unless both `EXECUTE_MOVES = True` and `EXECUTION_CONFIRMATION = "MOVE_FILES"` are set.


## 1. Imports and support-module discovery

In [ ]:

from __future__ import annotations

import importlib.util
import re
import sys
import warnings
from pathlib import Path
from typing import Dict, List

import pandas as pd
from IPython.display import display, HTML

display(HTML("<style>.container { width:100% !important; }</style>"))
warnings.filterwarnings("default")

from vip_slap2_analysis.utils import reorganize_slap2_session as reorg, directory_manifest as manifest_utils

## 2. Configuration

In [ ]:

# -----------------------------------------------------------------------------
# SESSION SELECTION
# -----------------------------------------------------------------------------
# "single": process exactly SESSION_DIR
# "subject_batch": find all raw sessions belonging to SUBJECT_ID under SUBJECT_ROOT
MODE = "single"

# Single-session mode example:
SESSION_DIR = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP8\852835\852835_2026-07-27_11-59-54")

# Subject-batch mode example:
SUBJECT_ROOT = SESSION_DIR.parent
SUBJECT_ID = "852835"  # exactly six digits

# -----------------------------------------------------------------------------
# REORGANIZATION OPTIONS
# -----------------------------------------------------------------------------
# "nested" is recommended. "sibling" reproduces the older behavior.
LAYOUT = "nested"

CLEANUP_EMPTY_DIRS = True
INCLUDE_HIDDEN_IN_MANIFEST = False
MAX_PREVIEW_ROWS = 200

# -----------------------------------------------------------------------------
# EXECUTION GATE
# -----------------------------------------------------------------------------
# Leave these values unchanged for the first review pass.
EXECUTE_MOVES = True
EXECUTION_CONFIRMATION = "MOVE_FILES"  # set exactly to "MOVE_FILES" when ready


## 3. Resolve and validate the selected sessions

In [ ]:

SESSION_NAME_RE = re.compile(r"(?P<subject>\d{6})_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}")


def resolve_sessions() -> List[Path]:
    if MODE == "single":
        session = SESSION_DIR.expanduser().resolve()
        if not session.is_dir():
            raise NotADirectoryError(f"SESSION_DIR does not exist: {session}")
        match = SESSION_NAME_RE.fullmatch(session.name)
        if match is None:
            raise ValueError(
                "SESSION_DIR must be named <six-digit-subject>_YYYY-MM-DD_HH-MM-SS; "
                f"received: {session.name}"
            )
        if not reorg.has_core_acquisition_content(session):
            raise ValueError(
                "The selected directory does not contain a raw acquisition folder such as "
                "behavior, behavior-videos, slap2, or imaging_data. It may already be organized."
            )
        return [session]

    if MODE == "subject_batch":
        root = SUBJECT_ROOT.expanduser().resolve()
        if not root.is_dir():
            raise NotADirectoryError(f"SUBJECT_ROOT does not exist: {root}")
        if re.fullmatch(r"\d{6}", SUBJECT_ID) is None:
            raise ValueError(f"SUBJECT_ID must contain exactly six digits: {SUBJECT_ID!r}")

        sessions = reorg.find_session_dirs(root, subject_ids={SUBJECT_ID})
        if not sessions:
            raise FileNotFoundError(
                f"No unorganized raw SLAP2 sessions for subject {SUBJECT_ID} were found under {root}"
            )
        return sessions

    raise ValueError("MODE must be either 'single' or 'subject_batch'")


sessions = resolve_sessions()
selection_df = pd.DataFrame({
    "subject_id": [SESSION_NAME_RE.fullmatch(path.name).group("subject") for path in sessions],
    "session_name": [path.name for path in sessions],
    "session_path": [str(path) for path in sessions],
})

print(f"Selected {len(sessions)} session(s).")
display(selection_df)



## 4. Write pre-reorganization tree manifests

Manifests are saved under each session's hidden `.reorganization_reports` directory. Because hidden paths are excluded by default, the report directory does not recursively appear in the manifest.


In [ ]:

def manifest_output_path(session: Path, phase: str) -> Path:
    report_dir = session / reorg.REPORT_DIR_NAME
    return report_dir / f"{session.name}_{phase}_tree_manifest.tsv"


manifest_rows = []
for session in sessions:
    output_path = manifest_output_path(session, "before")
    rows = manifest_utils.generate_directory_manifest(
        session,
        out_path=output_path,
        include_root=True,
        include_hidden=INCLUDE_HIDDEN_IN_MANIFEST,
        relative=True,
    )
    n_files = sum(kind == "file" for _, kind in rows)
    n_dirs = sum(kind == "dir" for _, kind in rows)
    manifest_rows.append({
        "session": session.name,
        "files": n_files,
        "directories": n_dirs,
        "manifest": str(output_path),
    })

pre_manifest_df = pd.DataFrame(manifest_rows)
display(pre_manifest_df)


## 5. Build, validate, and save dry-run move plans

In [ ]:

def plan_to_dataframe(plan) -> pd.DataFrame:
    rows = []
    for rec in plan.records:
        rows.append({
            "session": plan.target_session_dir.name,
            "category": rec.category,
            "status": rec.status,
            "source": str(rec.src),
            "destination": str(rec.dst),
            "reason": rec.reason,
        })
    return pd.DataFrame(rows)


plans: Dict[Path, object] = {}
plan_frames = []
summary_rows = []
validation_failures = []

for session in sessions:
    try:
        plan = reorg.build_reorganization_plan(session, layout=LAYOUT)
        errors = reorg.validate_plan(plan)
    except Exception as exc:
        validation_failures.append({"session": session.name, "error": repr(exc)})
        continue

    if errors:
        validation_failures.extend(
            {"session": session.name, "error": error} for error in errors
        )
        continue

    # Mark records as DRY_RUN and write the same operation report used by the CLI.
    reorg.execute_plan(plan, execute=False)
    report_path = (
        session / reorg.REPORT_DIR_NAME /
        f"{session.name}_slap2_reorganization_dry_run_report.tsv"
    )
    reorg.write_report(plan, report_path)

    frame = plan_to_dataframe(plan)
    plans[session] = plan
    plan_frames.append(frame)
    summary_rows.append({
        "session": session.name,
        "raw": int((frame["category"] == "raw").sum()) if not frame.empty else 0,
        "processed": int((frame["category"] == "processed").sum()) if not frame.empty else 0,
        "overflow": int((frame["category"] == "overflow").sum()) if not frame.empty else 0,
        "warnings": " | ".join(plan.warnings),
        "dry_run_report": str(report_path),
    })

if validation_failures:
    display(pd.DataFrame(validation_failures))
    raise RuntimeError("At least one session failed plan construction or validation. No execution is allowed.")

plan_df = pd.concat(plan_frames, ignore_index=True) if plan_frames else pd.DataFrame()
summary_df = pd.DataFrame(summary_rows)

print("Dry-run planning complete.")
display(summary_df)



## 6. Review planned moves

Pay particular attention to the overflow table. Unrecognized files are intentionally routed there instead of being deleted.


In [ ]:

if plan_df.empty:
    print("No planned moves.")
else:
    print(f"Showing up to {MAX_PREVIEW_ROWS} of {len(plan_df)} planned operations:")
    display(plan_df.head(MAX_PREVIEW_ROWS))

    overflow_df = plan_df.loc[plan_df["category"] == "overflow"].copy()
    print(f"Overflow operations: {len(overflow_df)}")
    if overflow_df.empty:
        print("No files are assigned to overflow.")
    else:
        display(overflow_df.head(MAX_PREVIEW_ROWS))



## 7. Execute the reorganization

Run the notebook once with execution disabled. Review the dry-run reports and overflow table. Only then set:

```python
EXECUTE_MOVES = True
EXECUTION_CONFIRMATION = "MOVE_FILES"
```

and rerun from the configuration cell onward. Immediately before moving anything, this cell rebuilds and revalidates every plan and refuses to start if a source is missing or a destination already exists.


In [ ]:

def find_execution_blockers(plan) -> List[str]:
    blockers = []
    for rec in plan.records:
        if not rec.src.exists():
            blockers.append(f"Missing source: {rec.src}")
        if rec.dst.exists():
            blockers.append(f"Destination already exists: {rec.dst}")
    return blockers


execution_rows = []
executed_plans: Dict[Path, object] = {}

if not EXECUTE_MOVES:
    print("DRY RUN ONLY: no files were moved.")
    print('To execute, set EXECUTE_MOVES=True and EXECUTION_CONFIRMATION="MOVE_FILES".')
else:
    if EXECUTION_CONFIRMATION != "MOVE_FILES":
        raise RuntimeError(
            'Execution was requested, but EXECUTION_CONFIRMATION is not exactly "MOVE_FILES".'
        )

    # Preflight every session before moving the first file, reducing the chance
    # of a partially completed batch caused by a later validation failure.
    fresh_plans: Dict[Path, object] = {}
    all_blockers = []
    for session in sessions:
        fresh_plan = reorg.build_reorganization_plan(session, layout=LAYOUT)
        errors = reorg.validate_plan(fresh_plan)
        blockers = errors + find_execution_blockers(fresh_plan)
        if blockers:
            all_blockers.extend(
                {"session": session.name, "blocker": blocker} for blocker in blockers
            )
        fresh_plans[session] = fresh_plan

    if all_blockers:
        display(pd.DataFrame(all_blockers))
        raise RuntimeError("Execution preflight failed. No files were moved.")

    for session, plan in fresh_plans.items():
        reorg.execute_plan(plan, execute=True)
        if CLEANUP_EMPTY_DIRS:
            reorg.cleanup_empty_dirs(session, execute=True)

        report_path = (
            session / reorg.REPORT_DIR_NAME /
            f"{session.name}_slap2_reorganization_executed_report.tsv"
        )
        reorg.write_report(plan, report_path)
        executed_plans[session] = plan

        counts = pd.Series([rec.status for rec in plan.records]).value_counts().to_dict()
        execution_rows.append({
            "session": session.name,
            "moved": counts.get("MOVED", 0),
            "destination_exists": counts.get("DEST_EXISTS", 0),
            "missing_source": counts.get("MISSING_SOURCE", 0),
            "report": str(report_path),
        })

    execution_df = pd.DataFrame(execution_rows)
    display(execution_df)

    failures = execution_df[execution_df["moved"] == 0] if not execution_df.empty else execution_df
    if any(row["destination_exists"] or row["missing_source"] for row in execution_rows):
        raise RuntimeError("One or more move operations failed. Inspect the executed reports.")

    print("Reorganization completed.")


## 8. Write post-reorganization manifests

In [ ]:

if not EXECUTE_MOVES:
    print("Post manifests are generated only after actual execution.")
else:
    post_rows = []
    for session in sessions:
        output_path = manifest_output_path(session, "after")
        rows = manifest_utils.generate_directory_manifest(
            session,
            out_path=output_path,
            include_root=True,
            include_hidden=INCLUDE_HIDDEN_IN_MANIFEST,
            relative=True,
        )
        post_rows.append({
            "session": session.name,
            "files": sum(kind == "file" for _, kind in rows),
            "directories": sum(kind == "dir" for _, kind in rows),
            "manifest": str(output_path),
        })

    post_manifest_df = pd.DataFrame(post_rows)
    display(post_manifest_df)



## Recommended use

### One session

Set `MODE = "single"` and edit `SESSION_DIR`. Run through the dry-run review first, then enable the execution gate.

### All sessions for one subject

Set `MODE = "subject_batch"`, point `SUBJECT_ROOT` to the directory containing subject/session folders, and set the six-digit `SUBJECT_ID`. Recursive discovery excludes canonical raw children inside already-organized nested sessions.

### Report locations

Each outer session container receives:

```text
.reorganization_reports/
    <session>_before_tree_manifest.tsv
    <session>_slap2_reorganization_dry_run_report.tsv
    <session>_slap2_reorganization_executed_report.tsv
    <session>_after_tree_manifest.tsv
```

The dry-run and executed reports list every source, destination, category, routing reason, and operation status.
